# 랜덤포레스트 — zoy.py 전처리 + KNN 노트북과 같은 조건

- **전처리**: `zoy.py`의 `preprocess()` 그대로 사용
- **분할·시드**: KNN 노트북과 동일 (`train_test_split(test_size=0.2, random_state=42, stratify=y)`, SEED=42)
- **결측 처리**: KNN 노트북처럼 `SimpleImputer(median)`을 거침 (zoy.py에서 이미 채워져서 실제로는 바뀌는 값 없음)
- **표준화**: 랜덤포레스트는 값의 크기에 영향을 받지 않아서 생략
- **모델**: `RandomForestClassifier(n_estimators=2000, class_weight="balanced", random_state=42)` — model_test.ipynb와 같은 설정

> 실행 위치: `학생배포` 폴더 (zoy.py가 `data/주조공정_입력.csv` 등을 상대경로로 읽음)

## 1. 경로 설정 + zoy.py 불러오기
`ZOY_DIR`만 실제 zoy.py가 있는 폴더로 맞춰주세요.

In [1]:
import os
import sys

# zoy.py가 있는 testpy 폴더를 import 경로에 추가
sys.path.append(os.getcwd())
import zoy

# zoy.py가 data/... 를 학생배포 기준으로 읽으므로 실행 위치를 한 칸 위로 이동
if os.path.basename(os.getcwd()) == "testpy":
    os.chdir("..")
print("현재 실행 위치:", os.getcwd())

현재 실행 위치: c:\Users\apxkz\OneDrive\Desktop\final_project\학생배포


## 2. 시드 고정 + 전처리

In [2]:
import random
import numpy as np
import pandas as pd

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

d = zoy.preprocess()
X = d["X_train"]
y = d["y_train"]
groups = d["groups"]

print("X:", X.shape, "| 피처", len(X.columns), "개")
print("라벨 분포:", y.value_counts().to_dict())

X: (5257, 23) | 피처 23 개
라벨 분포: {'정상': 4076, '불량_Short_Shot': 469, '불량_Blow_Hole': 233, '불량_Exfoliation': 197, '불량_Deformation': 106, '불량_Stain': 100, '기타불량': 76}


## 3. 분할 (KNN 노트북과 동일)

In [3]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=SEED,
    stratify=y,
)
print(f"학습: {len(X_train)}개 / 검증: {len(X_test)}개")

학습: 4205개 / 검증: 1052개


## 4. 랜덤포레스트 학습 + 검증

In [4]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report

def make_rf():
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("rf", RandomForestClassifier(
            n_estimators=2000,
            class_weight="balanced",
            random_state=SEED,
            n_jobs=-1,
        )),
    ])

rf = make_rf().fit(X_train, y_train)
pred_rf = rf.predict(X_test)

print("[랜덤포레스트 검증 결과]")
print(f"정확도: {accuracy_score(y_test, pred_rf):.4f}")
print(f"Macro F1: {f1_score(y_test, pred_rf, average='macro', zero_division=0):.4f}")
print(classification_report(y_test, pred_rf, digits=4, zero_division=0))

[랜덤포레스트 검증 결과]
정확도: 0.7234
Macro F1: 0.4532
                precision    recall  f1-score   support

          기타불량     0.1000    0.0667    0.0800        15
  불량_Blow_Hole     0.4118    0.8936    0.5638        47
불량_Deformation     0.1587    0.4762    0.2381        21
불량_Exfoliation     0.3333    0.6154    0.4324        39
 불량_Short_Shot     0.4878    0.6383    0.5530        94
      불량_Stain     0.3871    0.6000    0.4706        20
            정상     0.9401    0.7500    0.8344       816

      accuracy                         0.7234      1052
     macro avg     0.4027    0.5772    0.4532      1052
  weighted avg     0.8155    0.7234    0.7527      1052



## 5. 혼동 행렬 (행 = 실제, 열 = 예측)

In [5]:
LABELS = sorted(y.unique())
cm = pd.crosstab(y_test, pd.Series(pred_rf, index=y_test.index), rownames=["실제"], colnames=["예측"])
cm.reindex(index=LABELS, columns=LABELS, fill_value=0)

예측,기타불량,불량_Blow_Hole,불량_Deformation,불량_Exfoliation,불량_Short_Shot,불량_Stain,정상
실제,,,,,,,
기타불량,1,0,2,2,1,0,9
불량_Blow_Hole,0,42,0,0,3,1,1
불량_Deformation,1,0,10,4,2,0,4
불량_Exfoliation,1,0,5,24,1,0,8
불량_Short_Shot,0,4,5,4,60,5,16
불량_Stain,0,3,0,0,4,12,1
정상,7,53,41,38,52,13,612


## 6. 쌍둥이 효과 확인
검증 행을 **같은 샷(쌍둥이)이 학습 쪽에 있는 행**과 **처음 보는 행**으로 나눠서 점수를 봐요.

In [6]:
twin = groups.loc[X_test.index].isin(set(groups.loc[X_train.index]))
pred_s = pd.Series(pred_rf, index=X_test.index)

for name, m in [("쌍둥이가 학습에 있음", twin), ("처음 보는 샷", ~twin)]:
    print(f"{name:12s} {m.sum():4d}행 | 정확도 {accuracy_score(y_test[m], pred_s[m]):.4f} "
          f"| Macro F1 {f1_score(y_test[m], pred_s[m], average='macro', zero_division=0):.4f}")

쌍둥이가 학습에 있음   469행 | 정확도 0.8785 | Macro F1 0.8034
처음 보는 샷       583행 | 정확도 0.5986 | Macro F1 0.2925


## 7. 그룹 분할 점수 (같은 샷을 한쪽에만 두고 나눔)
쌍둥이 효과가 빠진, 처음 보는 샷에 대한 점수예요.

In [7]:
from sklearn.model_selection import GroupShuffleSplit

tr_idx, te_idx = next(GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED).split(X, y, groups))
rf_g = make_rf().fit(X.iloc[tr_idx], y.iloc[tr_idx])
pred_g = rf_g.predict(X.iloc[te_idx])

print(f"[그룹 분할] 학습 {len(tr_idx)} / 검증 {len(te_idx)}")
print(f"정확도: {accuracy_score(y.iloc[te_idx], pred_g):.4f}")
print(f"Macro F1: {f1_score(y.iloc[te_idx], pred_g, average='macro', zero_division=0):.4f}")

[그룹 분할] 학습 4236 / 검증 1021
정확도: 0.6298
Macro F1: 0.3428


## 8. 메모
- 무작위 분할 Macro F1:
- 쌍둥이 있음 / 처음 보는 샷:
- 그룹 분할 Macro F1:
- KNN과 비교:

In [10]:
# ===== 9. 성능 개선 실험: 같은 분할(X_train/X_test)에서 비교 =====
from sklearn.ensemble import ExtraTreesClassifier, HistGradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

N_TREES = 1000
ENV_COLS = ["Melting_Furnace_Temp", "Air_Pressure", "Coolant_Temp",
            "Coolant_Pressure", "Factory_Temp", "Factory_Humidity"]

MODELS = {
    "RF_balanced": lambda: RandomForestClassifier(n_estimators=N_TREES, class_weight="balanced",
                                                  random_state=SEED, n_jobs=-1),
    "RF_balanced_subsample": lambda: RandomForestClassifier(n_estimators=N_TREES, class_weight="balanced_subsample",
                                                            random_state=SEED, n_jobs=-1),
    "ExtraTrees": lambda: ExtraTreesClassifier(n_estimators=N_TREES, class_weight="balanced",
                                               random_state=SEED, n_jobs=-1),
    "KNN": lambda: Pipeline([("scaler", StandardScaler()),
                             ("knn", KNeighborsClassifier(n_neighbors=7, weights="distance"))]),
    "HistGB": lambda: HistGradientBoostingClassifier(class_weight="balanced", random_state=SEED),
}

results = []
def record(name, pred):
    f1 = f1_score(y_test, pred, average="macro", zero_division=0)
    acc = accuracy_score(y_test, pred)
    results.append({"실험": name, "정확도": round(acc, 4), "Macro F1": round(f1, 4)})
    print(f"{name:32s} 정확도 {acc:.4f} | Macro F1 {f1:.4f}")

# (1) 모델별 단독 성능
for name, make in MODELS.items():
    record(name, make().fit(X_train, y_train).predict(X_test))

# (2) 아이디어: 제품(Type1, Type2)별로 따로 학습
def fit_predict_per_type(make, Xa, ya, Xb):
    out = pd.Series(index=Xb.index, dtype=object)
    for t in [1, 2]:
        ma, mb = Xa["Product_Type"] == t, Xb["Product_Type"] == t
        out[mb] = make().fit(Xa[ma], ya[ma]).predict(Xb[mb])
    return out.values

record("RF_balanced 제품별", fit_predict_per_type(MODELS["RF_balanced"], X_train, y_train, X_test))
record("RF_balanced_subsample 제품별", fit_predict_per_type(MODELS["RF_balanced_subsample"], X_train, y_train, X_test))

# (3) 아이디어: 환경변수로 가동 구간을 묶은 '구간 번호'를 변수로 추가 (KMeans, 학습 데이터로만 학습)
env_scaler = StandardScaler().fit(X_train[ENV_COLS])
km = KMeans(n_clusters=7, n_init=10, random_state=SEED).fit(env_scaler.transform(X_train[ENV_COLS]))
X_train_c = X_train.assign(env_cluster=km.labels_)
X_test_c = X_test.assign(env_cluster=km.predict(env_scaler.transform(X_test[ENV_COLS])))
record("RF_balanced + 환경 구간 번호", MODELS["RF_balanced"]().fit(X_train_c, y_train).predict(X_test_c))

pd.DataFrame(results).sort_values("Macro F1", ascending=False).reset_index(drop=True)

RF_balanced                      정확도 0.7234 | Macro F1 0.4560
RF_balanced_subsample            정확도 0.8622 | Macro F1 0.5489
ExtraTrees                       정확도 0.8612 | Macro F1 0.5635
KNN                              정확도 0.8546 | Macro F1 0.5749
HistGB                           정확도 0.8089 | Macro F1 0.5176
RF_balanced 제품별                  정확도 0.7662 | Macro F1 0.4899
RF_balanced_subsample 제품별        정확도 0.8631 | Macro F1 0.5537
RF_balanced + 환경 구간 번호           정확도 0.7158 | Macro F1 0.4527


,실험,정확도,Macro F1
0,KNN,0.8546,0.5749
1,ExtraTrees,0.8612,0.5635
2,RF_balanced_subsample 제품별,0.8631,0.5537
3,RF_balanced_subsample,0.8622,0.5489
4,HistGB,0.8089,0.5176
5,RF_balanced 제품별,0.7662,0.4899
6,RF_balanced,0.7234,0.4560
7,RF_balanced + 환경 구간 번호,0.7158,0.4527


In [11]:
# ===== 10. 최종: 4개 모델 확률 평균(소프트 보팅) =====
ENSEMBLE = ["RF_balanced_subsample", "ExtraTrees", "KNN", "HistGB"]
CLASSES = np.array(sorted(y.unique()))

def ensemble_fit_predict(Xa, ya, Xb):
    probs = []
    for name in ENSEMBLE:
        m = MODELS[name]().fit(Xa, ya)
        p = pd.DataFrame(m.predict_proba(Xb), columns=m.classes_).reindex(columns=CLASSES, fill_value=0)
        probs.append(p.values)
    return CLASSES[np.mean(probs, axis=0).argmax(axis=1)]

pred_ens = ensemble_fit_predict(X_train, y_train, X_test)
record("앙상블 (RF_bsub+ET+KNN+HGB)", pred_ens)

print("\n[앙상블 검증 결과]")
print(f"정확도: {accuracy_score(y_test, pred_ens):.4f}")
print(f"Macro F1: {f1_score(y_test, pred_ens, average='macro', zero_division=0):.4f}")
print(classification_report(y_test, pred_ens, digits=4, zero_division=0))

cm = pd.crosstab(y_test, pd.Series(pred_ens, index=y_test.index), rownames=["실제"], colnames=["예측"])
cm.reindex(index=CLASSES, columns=CLASSES, fill_value=0)

앙상블 (RF_bsub+ET+KNN+HGB)         정확도 0.8593 | Macro F1 0.5718

[앙상블 검증 결과]
정확도: 0.8593
Macro F1: 0.5718
                precision    recall  f1-score   support

          기타불량     1.0000    0.0667    0.1250        15
  불량_Blow_Hole     0.8333    0.7447    0.7865        47
불량_Deformation     0.4545    0.2381    0.3125        21
불량_Exfoliation     0.6970    0.5897    0.6389        39
 불량_Short_Shot     0.6866    0.4894    0.5714        94
      불량_Stain     0.7857    0.5500    0.6471        20
            정상     0.8857    0.9596    0.9212       816

      accuracy                         0.8593      1052
     macro avg     0.7633    0.5197    0.5718      1052
  weighted avg     0.8497    0.8593    0.8447      1052



예측,기타불량,불량_Blow_Hole,불량_Deformation,불량_Exfoliation,불량_Short_Shot,불량_Stain,정상
실제,,,,,,,
기타불량,1,0,0,0,0,0,14
불량_Blow_Hole,0,35,0,0,1,0,11
불량_Deformation,0,0,5,0,0,0,16
불량_Exfoliation,0,0,0,23,1,0,15
불량_Short_Shot,0,2,1,2,46,2,41
불량_Stain,0,0,0,0,5,11,4
정상,0,5,5,8,14,1,783


In [12]:
# ===== 11. 점검: 쌍둥이 효과 / 그룹 분할 (7번 셀과 같은 분할) =====
pred_s = pd.Series(pred_ens, index=X_test.index)
for name, m in [("쌍둥이가 학습에 있음", twin), ("처음 보는 샷", ~twin)]:
    print(f"{name:12s} {m.sum():4d}행 | Macro F1 {f1_score(y_test[m], pred_s[m], average='macro', zero_division=0):.4f}")

pred_g_ens = ensemble_fit_predict(X.iloc[tr_idx], y.iloc[tr_idx], X.iloc[te_idx])
print(f"[그룹 분할] 앙상블 Macro F1: {f1_score(y.iloc[te_idx], pred_g_ens, average='macro', zero_division=0):.4f} "
      f"(7번 셀 RF: {f1_score(y.iloc[te_idx], pred_g, average='macro', zero_division=0):.4f})")

쌍둥이가 학습에 있음   469행 | Macro F1 0.9682
처음 보는 샷       583행 | Macro F1 0.2982
[그룹 분할] 앙상블 Macro F1: 0.3173 (7번 셀 RF: 0.3428)
